In [ ]:
!pip install mediapipe gradio
import mediapipe as mp
import cv2
import numpy as np
import gradio as gr

# Drawing helpers
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

In [ ]:
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

In [ ]:
def check_lunge_feedback(landmarks):
    # Front Knee Angle (Left side: Hip-Knee-Ankle)
    left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_ankle = [landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    # Back Knee Angle (Right side: Hip-Knee-Ankle)
    right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
    right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
    right_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

    front_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    back_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)

    # --- Initialize feedback variables ---
    feedback_front = ""
    feedback_back = ""

    # Check 1: Front Knee Depth
    if front_knee_angle < 80:
        feedback_front = "Too Deep! Front Knee Dangerously Low" # Corrected line: Assigning to feedback_front
    elif 80 <= front_knee_angle <= 100:
        feedback_front = "Good Front Knee Angle"
    elif 100 < front_knee_angle <= 140:
        feedback_front = "Go Lower! Front Knee Angle too High"
    else:
        feedback_front = "Too High! Bend Front Knee More"

    # Check 2: Back Knee Depth
    if back_knee_angle < 80:
        feedback_back = "Back Knee Too Close to Floor" # Corrected line: Assigning to feedback_back
    elif 80 <= back_knee_angle <= 100:
        feedback_back = "Good Back Knee Angle"
    elif 100 < back_knee_angle <= 140:
        feedback_back = "Go Lower! Back Knee Angle too High"
    else:
        feedback_back = "Too High! Bend Back Knee More"

    # Combine Feedback
    if "Good" in feedback_front and "Good" in feedback_back:
        overall_feedback = "Perfect Lunge Form!"
    else:
        overall_feedback = f"Front Knee: {feedback_front} | Back Knee: {feedback_back}"


    # --- Accuracy Calculation (Target angle 90 degrees for both) ---
    front_score = max(0, 100 - abs(front_knee_angle - 90) * 3)
    back_score = max(0, 100 - abs(back_knee_angle - 90) * 3)

    accuracy = (front_score + back_score) / 2
    accuracy = max(0, min(100, int(accuracy)))

    return overall_feedback, int(accuracy)

In [ ]:
def analyze_lunges(video_path):
    cap = cv2.VideoCapture(video_path)
    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))

    # Fix for video speed (stable FPS)
    fps = 30

    # CORRECTED LINE: Use the * operator to unpack the four characters
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    output_video = "output_lunge.mp4"
    out = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Fix for mirrored video (un-mirroring webcam view)
        frame = cv2.flip(frame, 1)

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            landmarks = results.pose_landmarks.landmark

            feedback, accuracy = check_lunge_feedback(landmarks)

            color = (0, 255, 0) if "Perfect" in feedback else (0, 0, 255)

            cv2.putText(image, feedback, (50, 50), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)
            cv2.putText(image, f"Accuracy: {accuracy}%", (50, 100), cv2.FONT_HERSHEY_COMPLEX, 1, color, 3)

        out.write(image)

    cap.release()
    out.release()
    return output_video

# Gradio Interface
gr.Interface(
    fn=analyze_lunges,
    inputs=gr.Video(sources=["upload", "webcam"]),
    outputs=gr.Video(),
    title="Lunge Form Analyzer",
    description="Upload a video of your lunges, and get feedback on your form!",
).launch(share=True, debug=True)